In [2]:
!pip install "granite-tsfm[notebooks] @ git+https://github.com/ibm-granite/granite-tsfm.git@v0.2.22"
!pip install numpy==1.25.2
!pip install --upgrade --force-reinstall pandas

  Cloning https://github.com/ibm-granite/granite-tsfm.git (to revision v0.2.22) to /tmp/pip-install-6py5kbiw/granite-tsfm_1e6f2dd46c1b4f28aab0dfc6b32cb81e
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite/granite-tsfm.git /tmp/pip-install-6py5kbiw/granite-tsfm_1e6f2dd46c1b4f28aab0dfc6b32cb81e
  Running command git checkout -q 216850d0cb073e31689049c1334f701fe11bc2c3
  Resolved https://github.com/ibm-granite/granite-tsfm.git to commit 216850d0cb073e31689049c1334f701fe11bc2c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 27.3 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2
    Uninstalling tzdata-2025.2:
      Successfully uninstalled tzdata-2025.2
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six

In [1]:
import os
import tempfile

from transformers import Trainer, TrainingArguments, set_seed

from tsfm_public import TinyTimeMixerForPrediction, load_dataset
from tsfm_public.toolkit import RecursivePredictor, RecursivePredictorConfig
from tsfm_public.toolkit.visualization import plot_predictions
from tsfm_public import TimeSeriesPreprocessor, get_datasets
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import glob

In [4]:
import torch
from copy import deepcopy

def update_past_values_with_predictions(dset_test, predictions, context_length=512):
  from copy import deepcopy
  import torch

  samples = []

  for i in range(len(dset_test)):
    sample = deepcopy(dset_test[i])
    old_past = sample['past_values']        # shape [512, 1]
    new_vals_np = predictions[i]                     # shape [96, 1] expected

    new_vals = torch.tensor(new_vals_np, dtype=old_past.dtype)

    combined = torch.cat([old_past, new_vals], dim=0)[-context_length:]

    sample["past_values"] = combined
    samples.append(sample)

  return samples

In [5]:
frac = 75
location = "uniform"
temp_dir = tempfile.mkdtemp()

In [6]:
n=9

In [7]:
# fetch the model
model_path = f"./drive/MyDrive/ColabBackup/vary_n_models/n_{n}/output"
matching_dirs = glob.glob(os.path.join(model_path, "checkpoint-*"))
checkpoint_path = matching_dirs[0]
model = TinyTimeMixerForPrediction.from_pretrained(checkpoint_path)

df = pd.read_parquet("./drive/MyDrive/ColabBackup/MGdatasets/data_vary_n.parquet")

X = np.array(df[f'{n}'].values)
T = np.arange(len(X))

# data preprocessing
MGdata = pd.DataFrame({
    'time': [round(x) for x in T],
    'P': ((X-np.mean(X))/(np.var(X)**(0.5))),
})

timestamp_column_MG = "time"
id_columns_MG = []  # mention the ids that uniquely identify a time-series.

target_columns_MG = ["P"]
split_config_MG = {
    "train": [0, 6000],
    "valid": [6000, 9000],
    "test": [
        9000,
        15001,
    ],
}
column_specifiers_MG = {
    "timestamp_column": timestamp_column_MG,
    "id_columns": id_columns_MG,
    "target_columns": target_columns_MG,
    "control_columns": [],
}
tsp = TimeSeriesPreprocessor(
    **column_specifiers_MG,
    context_length=512,
    prediction_length=96,                          # match your time interval
    scaling=False,
    encode_categorical=False,
    #scaler_type="standard",           # match training
)

_, _, dset_test = get_datasets(
    tsp, MGdata, split_config_MG,
    fewshot_fraction=1.0,
    fewshot_location=location
)


# model init
zeroshot_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=temp_dir,
        per_device_eval_batch_size=64,
        seed = 0,
    ),
)



In [8]:
dset_test_sc =[]
for i in range(400):
  data_test_sc = deepcopy(dset_test[i*10])
  data_test_sc['past_values'] = dset_test[i*10]['past_values']
  data_test_sc['future_values'] = dset_test[i*10]['future_values']
  dset_test_sc.append(data_test_sc)

for i in range(100):
  data_test_sc = deepcopy(dset_test[0])
  data_test_sc['past_values'] = dset_test[0]['past_values']
  data_test_sc['future_values'] = dset_test[0]['future_values']
  data_test_sc['past_values'][511] = data_test_sc['past_values'][511]-0.5+0.01*i
  dset_test_sc.append(data_test_sc)

# first predction
predictions_dict_first = zeroshot_trainer.predict(dset_test_sc)
pred = predictions_dict_first.predictions[0]

preds = pred
for i in range(11):
  dset_test_sc = update_past_values_with_predictions(dset_test_sc, pred)
  predictions_dict = zeroshot_trainer.predict(dset_test_sc)
  pred = predictions_dict.predictions[0]
  preds = np.concatenate((preds,pred),axis = 1)

preds = preds.squeeze(-1)

preds_TTM = preds[:,:1024,]

for i in range(500):
  preds_TTM[i] = preds_TTM[i]*np.var(X)**(0.5)+np.mean(X)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zl3340 (zl3340-columbia-university-in-the-city-of-new-york) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [9]:
df_preds = pd.DataFrame(preds_TTM)
df_preds['n'] = n

In [10]:
df_preds.to_parquet("preds.parquet")

/usr/local/lib/python3.11/dist-packages/pandas/io/parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
